In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest

In [2]:
#Load the users level table
users = pd.read_csv('../data/processed/user_level_analysis.csv')

#Inspect
users.head()

#Verify all user_ids are unique
assert users['user_id'].is_unique, 'User IDs are not unique'

#verify two variants and two segments
users['variant'].unique()
users['segment'].unique()

#Verify no null values
users.isna().sum()

#Inspect all numeric columns
users.describe()

,total_sessions,total_completed_sessions,total_abandoned_sessions,total_orders,total_spend,total_units
count,13059.000000,13059.000000,13059.000000,13059.000000,13059.000000,13059.000000
mean,2.068152,1.581208,0.486944,1.581208,78.922787,5.965082
std,1.621605,1.377861,0.729471,1.377861,197.419180,11.469525
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,1.000000,16.100000,1.000000
50%,1.000000,1.000000,0.000000,1.000000,33.990000,3.000000
75%,3.000000,2.000000,1.000000,2.000000,71.155000,6.000000
max,14.000000,14.000000,7.000000,14.000000,5244.340000,151.000000


In [ ]:
#Chi squared goodness of fit test to verify no Sample Ratio Mismatch.

#Null Hypothesis: Equal distribution of users in treatment and control
#Alternative Hypothesis: Unequal distribution of users in treatment and control

control = users[users['variant'] == 'control']
treatment = users[users['variant'] == 'treatment']
total = len(users)

observed = [len(control), len(treatment)]
expected = [total/2, total/2]

chi2, p_val = chisquare(f_obs=observed, f_exp=expected)

print(f'Chi-square statistic: {chi2:.4f}')
print(f'P-value: {p_val:.4f}') #P-value ~ 0.66.
#p > 0.05. Fail to reject null hypothesis.
# Treatment and control are equally distributed

Chi-square statistic: 0.1992
P-value: 0.6554


In [4]:
#Load orders level table
orders = pd.read_csv('../data/processed/order_level_analysis.csv')

#Inspect
orders.head()

#Verify two variants
orders['variant'].unique()

#Check all orders are unique
assert orders['order_id'].is_unique, 'Order IDs are not unique'

#Verify no null values
orders.isna().sum()

#Inspect numeric columns
orders.describe()

,order_amount,unit_count
count,20686.000000,20686.000000
mean,49.918329,3.774195
std,146.953452,8.207551
min,3.000000,1.000000
25%,17.190000,1.000000
50%,26.020000,2.000000
75%,39.660000,3.000000
max,4000.000000,79.000000


In [5]:
# Average Order Value

# Inspect aov and spread by variant
aov = orders.groupby('variant').agg(aov=('order_amount', 'mean'), aov_std=('order_amount', 'std'))
print(aov)
# control aov ~ 46.4, treatment aov ~ 53.6
# control order amount stdev ~ 133.84, treatment order amount stdev ~ 159.35

# Null hypothesis: ~ $7.2 gap is not caused by variant
# Alternative hypothesis: ~ $7.2 gap is due to treatment

control_amounts = np.array(orders[orders['variant'] == 'control']['order_amount'])
treatment_amounts = np.array(orders[orders['variant'] == 'treatment']['order_amount'])

t_stat, p_val = ttest_ind(control_amounts, treatment_amounts,equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0005
# Reject the null hypothesis.
# Average Order Value went up in treatment from $46.4 to $53.6, a ~ $7.2 increase.

                 aov     aov_std
variant                         
control    46.405776  133.842856
treatment  53.563053  159.346480
T-test statistic: -3.4917
P-value: 0.0005


In [ ]:
# Frequent vs occasional users Average Order Value

# Inspect aov and spread by variant among segments
aov = orders.groupby(['variant','segment']).agg(aov=('order_amount', 'mean'), aov_std=('order_amount', 'std'))
print(aov)
# control frequent aov ~ 46.3, treatment frequent aov ~ 55.2
# control occasional aov ~ 46.47, treatment occasional aov ~ 52.69

# Null hypothesis: ~ $8.9 increase is not caused by variant among frequent users
# Alternative hypothesis: ~ $8.9 increase is due to treatment among frequent users

freq_control_amounts = np.array(orders[(orders['variant'] == 'control') & (orders['segment'] == 'frequent')]['order_amount'])
freq_treatment_amounts = np.array(orders[(orders['variant'] == 'treatment') & (orders['segment'] == 'frequent')]['order_amount'])

t_stat, p_val = ttest_ind(freq_control_amounts, freq_treatment_amounts,equal_var=False)

print('Frequent Users')
print('--------------')
print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0104
# Reject the null hypothesis.
# Average Order Value went up in treatment among frequent users from $46.3 to $55.2, a ~ $8.9 increase.

#-------------------------------------------------------------------------------------------------------
print()
# Null hypothesis: ~ $6.22 increase is not caused by variant among frequent users
# Alternative hypothesis: ~ $6.22 increase is due to treatment among frequent users

occ_control_amounts = np.array(orders[(orders['variant'] == 'control') & (orders['segment'] == 'occasional')]['order_amount'])
occ_treatment_amounts = np.array(orders[(orders['variant'] == 'treatment') & (orders['segment'] == 'occasional')]['order_amount'])

t_stat, p_val = ttest_ind(occ_control_amounts, occ_treatment_amounts,equal_var=False)

print('Occasional Users')
print('--------------')
print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0142
# Reject the null hypothesis.
# Average Order Value also went up in treatment among occasional users from $46.47 to $52.69, a ~ $6.22 increase.

#Frequent shoppers show a bigger AOV lift ($46.3 to $55.2, ~+19%) than occasional shoppers ($46.5 to $52.7, ~+13%)

                            aov     aov_std
variant   segment                          
control   frequent    46.268105  121.324769
          occasional  46.478343  139.999897
treatment frequent    55.205953  168.694956
          occasional  52.693340  154.174338
Frequent Users
--------------
T-test statistic: -2.5645
P-value: 0.0104

Occasional Users
--------------
T-test statistic: -2.4524
P-value: 0.0142


In [7]:
#Units Per Transaction

#Inspect upt and spread by variant
upt = orders.groupby('variant').agg(upt=('unit_count', 'mean'), upt_std=('unit_count', 'std'))
print(upt)
#control upt ~ 3.48, treatment upt ~ 4.08
#control unit count stdev ~ 7.39, treatment unit count stdev ~ 8.97

#Null hypothesis: ~ 0.6 unit gap is not caused by variant
#Alternative hypothesis: ~ 0.6 unit gap is due to treatment

control_units = np.array(orders[orders['variant'] == 'control']['unit_count'])
treatment_units = np.array(orders[orders['variant'] == 'treatment']['unit_count'])

t_stat, p_val = ttest_ind(control_units, treatment_units, equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.000
# Reject the null hypothesis.
# Units Per Transaction went up in treatment from 3.48 to 4.08, a ~0.6 increase.

                upt   upt_std
variant                      
control    3.481583  7.390351
treatment  4.077817  8.967484
T-test statistic: -5.2081
P-value: 0.0000


In [8]:
#Revenue Per Revenue-Generating Users
rpr_users = users[users['total_spend'] != 0]

#Inspect rpru and spread by variant
rpru = rpr_users.groupby('variant').agg(rpru=('total_spend', 'mean'), rpru_std=('total_spend', 'std'))
#control rpru ~ 85.63, treatment rpru ~ 97.72
#control unit count stdev ~ 191.62, treatment unit count stdev ~ 227.06
print(rpru)

#Null hypothesis: ~ $12.09 gap is not caused by variant
#Alternative hypothesis: ~ $12.09 gap is due to treatment

control_spend = np.array(rpr_users[rpr_users['variant'] == 'control']['total_spend'])
treatment_spend = np.array(rpr_users[rpr_users['variant'] == 'treatment']['total_spend'])

t_stat, p_val = ttest_ind(control_spend, treatment_spend, equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0023
# Reject the null hypothesis.
# Revenue per revenue generating user went up in treatment from $85.63 to $97.72, a ~$12.09 increase.


                rpru    rpru_std
variant                         
control    85.632530  191.618912
treatment  97.718869  227.057876
T-test statistic: -3.0477
P-value: 0.0023


In [ ]:
# Frequent vs occasional Revenue Per Revenue-Generating Users

# Inspect rpru and spread by variant among segments
rpru_segmented = rpr_users.groupby(['variant','segment']).agg(rpru_segmented=('total_spend', 'mean'), rpru_std=('total_spend', 'std'))
print(rpru_segmented)
# control frequent rpru ~ 128.19, treatment frequent rpru ~ 151.49
# control occasional rpru ~ 72.94, treatment occasional rpru ~ 81.69

# Null hypothesis: ~ $23.3 increase is not caused by variant among frequent users
# Alternative hypothesis: ~ $23.3 increase is due to treatment among frequent users

freq_control_spend = np.array(rpr_users[(rpr_users['variant'] == 'control') & (rpr_users['segment'] == 'frequent')]['total_spend'])
freq_treatment_spend = np.array(rpr_users[(rpr_users['variant'] == 'treatment') & (rpr_users['segment'] == 'frequent')]['total_spend'])

t_stat, p_val = ttest_ind(freq_control_spend, freq_treatment_spend,equal_var=False)

print('Frequent Users')
print('--------------')
print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0209
# Reject the null hypothesis.
# Revenue per Revenue-Generating User went up in treatment from $128.19 to $151.49, a ~ $23.3 increase.

#-------------------------------------------------------------------------------------------------------
print()
# Null hypothesis: ~ $8.75 increase is not caused by variant among frequent users
# Alternative hypothesis: ~ $8.75 increase is due to treatment among frequent users

occ_control_spend = np.array(rpr_users[(rpr_users['variant'] == 'control') & (rpr_users['segment'] == 'occasional')]['total_spend'])
occ_treatment_spend = np.array(rpr_users[(rpr_users['variant'] == 'treatment') & (rpr_users['segment'] == 'occasional')]['total_spend'])

t_stat, p_val = ttest_ind(occ_control_spend, occ_treatment_spend,equal_var=False)

print('Occasional Users')
print('--------------')
print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0341
# Reject the null hypothesis.
# Revenue per Revenue-Generating User went up in treatment from $72.94 to $81.69, a ~ $8.75 increase.

#Frequent shoppers show a bigger RPRU lift ($128.19 to $151.49, ~+18%) than occasional shoppers ($72.94 to $81.69, ~+12%)

                      rpru_segmented    rpru_std
variant   segment                               
control   frequent        128.192735  222.012457
          occasional       72.942027  179.661508
treatment frequent        151.485278  285.499587
          occasional       81.694518  203.779374
Frequent Users
--------------
T-test statistic: -2.3110
P-value: 0.0209

Occasional Users
--------------
T-test statistic: -2.1191
P-value: 0.0341


In [10]:
# Abandonment Rate
total_control_sessions = control['total_sessions'].sum()
total_control_abandoned_sessions = control['total_abandoned_sessions'].sum()
control_abandonment_rate = total_control_abandoned_sessions/total_control_sessions
print(f'Control abandonment rate = {control_abandonment_rate * 100:.2f}%') #22.18%

total_treatment_sessions = treatment['total_sessions'].sum()
total_treatment_abandoned_sessions = treatment['total_abandoned_sessions'].sum()
treatment_abandonment_rate = total_treatment_abandoned_sessions/total_treatment_sessions
print(f'Treatment abandonment rate = {treatment_abandonment_rate * 100:.2f}%') #24.91%

# Null hypothesis: Abandonment rates across both variants are equal
# Alternative hypothesis: Abandonment rates are different in variants

abandonments = np.array([total_control_abandoned_sessions,total_treatment_abandoned_sessions])
sessions = np.array([total_control_sessions, total_treatment_sessions])

z_stat, p_val = proportions_ztest(count=abandonments, nobs=sessions)

print(f'Z-test statistic: {z_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p~value ~ 0.0000
# Reject null hypothesis. 
# Abandonment rate went up in treatment from 22.18% to 24.91%, a ~12% relative increase.

Control abandonment rate = 22.18%
Treatment abandonment rate = 24.91%
Z-test statistic: -5.2803
P-value: 0.0000


In [ ]:
#Abandonment rate of frequent vs occasional users
control_freq_users = users[(users['variant'] == 'control') & (users['segment'] == 'frequent')]
treatment_freq_users = users[(users['variant'] == 'treatment') & (users['segment'] == 'frequent')]

total_control_freq_sessions = control_freq_users['total_sessions'].sum()
total_control_freq_abandoned_sessions = control_freq_users['total_abandoned_sessions'].sum()
control_freq_abandonment_rate = total_control_freq_abandoned_sessions/total_control_freq_sessions
print(f'Control abandonment rate among frequent users = {control_freq_abandonment_rate * 100:.2f}%') #19.51%

total_treatment_freq_sessions = treatment_freq_users['total_sessions'].sum()
total_treatment_freq_abandoned_sessions = treatment_freq_users['total_abandoned_sessions'].sum()
treatment_freq_abandonment_rate = total_treatment_freq_abandoned_sessions/total_treatment_freq_sessions
print(f'Treatment abandonment rate among frequent users = {treatment_freq_abandonment_rate * 100:.2f}%') #21.45%

# Null hypothesis: Abandonment rates across both variants are equal among frequent users
# Alternative hypothesis: Abandonment rates are different in variants among frequent users

freq_abandonments = np.array([total_control_freq_abandoned_sessions,total_treatment_freq_abandoned_sessions])
freq_sessions = np.array([total_control_freq_sessions, total_treatment_freq_sessions])

z_stat, p_val = proportions_ztest(count=freq_abandonments, nobs=freq_sessions)

print(f'Z-test statistic: {z_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p~value ~ 0.0230
# Reject null hypothesis. 
# Abandonment rate went up in treatment among frequent users from 19.51% to 21.45%, a ~9.94% relative increase.



control_occ_users = users[(users['variant'] == 'control') & (users['segment'] == 'occasional')]
treatment_occ_users = users[(users['variant'] == 'treatment') & (users['segment'] == 'occasional')]

total_control_occ_sessions = control_occ_users['total_sessions'].sum()
total_control_occ_abandoned_sessions = control_occ_users['total_abandoned_sessions'].sum()
control_occ_abandonment_rate = total_control_occ_abandoned_sessions/total_control_occ_sessions
print(f'Control abandonment rate among occasional users = {control_occ_abandonment_rate * 100:.2f}%') #23.52%

total_treatment_occ_sessions = treatment_occ_users['total_sessions'].sum()
total_treatment_occ_abandoned_sessions = treatment_occ_users['total_abandoned_sessions'].sum()
treatment_occ_abandonment_rate = total_treatment_occ_abandoned_sessions/total_treatment_occ_sessions
print(f'Treatment abandonment rate among occasional users = {treatment_occ_abandonment_rate * 100:.2f}%') #26.62%

# Null hypothesis: Abandonment rates across both variants are equal among occasional users
# Alternative hypothesis: Abandonment rates are different in variants among occasional users

occ_abandonments = np.array([total_control_occ_abandoned_sessions,total_treatment_occ_abandoned_sessions])
occ_sessions = np.array([total_control_occ_sessions, total_treatment_occ_sessions])

z_stat, p_val = proportions_ztest(count=occ_abandonments, nobs=occ_sessions)

print(f'Z-test statistic: {z_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p~value ~ 0.0000
# Reject null hypothesis. 
# Abandonment rate went up in treatment among occasional users from 23.52% to 26.62%, a ~13.18% relative increase.

#Occasional shoppers abandoned their carts at a higher rate (23.52% to 26.62%, ~+13.2%) than frequent shoppers (19.51% to 21.45%, ~+9.9%)

Control abandonment rate among frequent users = 19.51%
Treatment abandonment rate among frequent users = 21.45%
Z-test statistic: -2.2730
P-value: 0.0230
Control abandonment rate among occasional users = 23.52%
Treatment abandonment rate among occasional users = 26.62%
Z-test statistic: -4.8002
P-value: 0.0000


In [ ]:
#Revenue Forecast -- Frequent Shoppers Only
COMPANY_USER_BASE = 200_000
annualization_factor = 365 / 90

# what fraction of the user base is "frequent"?
frequent_fraction = (users['segment'] == 'frequent').mean()
frequent_user_base = COMPANY_USER_BASE * frequent_fraction

print(f'Frequent shoppers: {frequent_fraction*100:.1f}% of user base')
print(f'Frequent shopper population (of {COMPANY_USER_BASE:,}): {frequent_user_base:,.0f}')

# average total spend per user, control vs treatment, frequent shoppers only
# (includes $0 for abandoners, same logic as the overall forecast)
freq_spend = users[users['segment'] == 'frequent'].groupby('variant')['total_spend'].mean()
control_freq_spend = freq_spend['control']
treatment_freq_spend = freq_spend['treatment']

# annualize and compute the lift, frequent shoppers only
baseline_freq_per_user = control_freq_spend * annualization_factor
treatment_freq_per_user = treatment_freq_spend * annualization_factor
lift_freq_per_user = treatment_freq_per_user - baseline_freq_per_user
lift_freq_pct = lift_freq_per_user / baseline_freq_per_user

# scale to only the frequent-shopper portion of the user base
#(this is the number consistent with a segment-targeted rollout,
#not the full 200K company user base)
annual_incremental_revenue_freq_only = lift_freq_per_user * frequent_user_base

print(f'Baseline (frequent) revenue/user/year: ${baseline_freq_per_user:.2f}')
print(f'Net lift/user/year (frequent only): ${lift_freq_per_user:.2f} (~{lift_freq_pct*100:.1f}%)')
print(f'Projected annual incremental revenue (frequent shoppers only): ${annual_incremental_revenue_freq_only:,.2f}')

Frequent shoppers: 20.2% of user base
Frequent shopper population (of 200,000): 40,493
Baseline (frequent) revenue/user/year: $510.15
Net lift/user/year (frequent only): $87.79 (~17.2%)
Projected annual incremental revenue (frequent shoppers only): $3,555,023.10


## Executive Summary

- Decision: Roll out to frequent shoppers; hold off on occasional shoppers
  pending further review.
- Primary metrics: AOV +15.4%, RPRU +14.1% overall (frequent shoppers +18%,
  occasional shoppers +12%).
- Guardrail: Cart abandonment increased significantly (22.18% → 24.91%,
  p < 0.0001), concentrated more heavily among occasional shoppers
  (+13.2% relative) than frequent shoppers (+9.9% relative).
- Estimated impact: ~$3.6M net annual incremental revenue if targeted to
  frequent shoppers (~20% of the user base), reflecting a 17.2% lift for
  that segment, net of the abandonment tradeoff. A full company-wide
  rollout would project ~$7.3M, but carries a higher abandonment cost from
  occasional shoppers.
- Recommendation: Deploy the carousel to frequent shoppers, where the
  spend lift is largest and the abandonment cost is smallest. Consider
  UX refinements (e.g. an easier skip/dismiss) before extending to
  occasional shoppers, and monitor abandonment post-launch.